In [3]:
import os
import requests
import json
from datetime import datetime, timedelta

### The Live Telemetry Endpoint

In [4]:

print("--- FETCHING LIVE AIRSPACE ---")
states_url = "https://opensky-network.org/api/states/all"

zrh_bbox = {
    "lamin": 46.21, "lamax": 48.71, 
    "lomin": 6.80, "lomax": 10.30
}

response = requests.get(states_url, params=zrh_bbox, timeout=10)

if response.status_code == 200:
    data = response.json()
    print(f"Total flights found: {len(data['states'])}")
    print("Sample plane:")
    print(data['states'][0])
else:
    print(f"Error {response.status_code}: {response.text}")

--- FETCHING LIVE AIRSPACE ---
Total flights found: 115
Sample plane:
['3d1289', 'DEFGR   ', 'Germany', 1789231443, 1789231443, 10.2488, 47.5905, 1394.46, False, 55.49, 72.74, 1.63, None, 1554.48, '4453', False, 0]


### The Historical Arrivals Log

In [3]:
import time

print("\n--- FETCHING ZURICH ARRIVALS ---")
arrivals_url = "https://opensky-network.org/api/flights/arrival"

# Calculate Unix timestamps for the last X days
days = 4
end_time = int(time.time())
begin_time = end_time - (days * 24 * 60 * 60) # X days ago in seconds

arrival_params = {
    "airport": "LSZH",
    "begin": begin_time,
    "end": end_time
}

arr_response = requests.get(arrivals_url, params=arrival_params, timeout=10)

if arr_response.status_code == 200:
    arrivals_data = arr_response.json()
    print(f"Total arrivals in the last {days} days: {len(arrivals_data)}")
    if len(arrivals_data) > 0:
        print("Sample Arrival Record:")
        print(json.dumps(arrivals_data[0], indent=2))
else:
    print(f"Error {arr_response.status_code}: {arr_response.text}")


--- FETCHING ZURICH ARRIVALS ---
Error 403: You cannot access historical flights


####  Authentication is required according to the documentation:
https://www.openskynetwork.github.io/opensky-api/rest.html#authentication 


In [5]:
## Authentication is required according to the documentation
import os
from dotenv import load_dotenv

# Load the credentials
load_dotenv()
CLIENT_ID = os.getenv("OPENSKY_CLIENT_ID")
CLIENT_SECRET = os.getenv("OPENSKY_CLIENT_SECRET")

class TokenManager:
    """Handles OAuth2 Token fetching and refreshing for OpenSky."""
    def __init__(self, client_id, client_secret):
        self.client_id = client_id
        self.client_secret = client_secret
        self.token_url = "https://auth.opensky-network.org/auth/realms/opensky-network/protocol/openid-connect/token"
        self.token = None
        self.expires_at = None
        self.refresh_margin = 30 # seconds

    def get_token(self):
        # If token exists and hasn't expired, return it
        if self.token and self.expires_at and datetime.now() < self.expires_at:
            return self.token
        # Otherwise, fetch a new one
        return self._refresh()

    def _refresh(self):
        r = requests.post(
            self.token_url,
            data={
                "grant_type": "client_credentials",
                "client_id": self.client_id,
                "client_secret": self.client_secret,
            },
        )
        r.raise_for_status()
        data = r.json()
        self.token = data["access_token"]
        expires_in = data.get("expires_in", 1800)
        self.expires_at = datetime.now() + timedelta(seconds=expires_in - self.refresh_margin)
        return self.token

    def headers(self):
        return {"Authorization": f"Bearer {self.get_token()}"}

# Initialize the manager
opensky_auth = TokenManager(CLIENT_ID, CLIENT_SECRET)

In [ ]:
## TESTING AUTHENTICATED REQUESTS

import requests

print("--- FETCHING AUTHENTICATED LIVE AIRSPACE ---")
states_url = "https://opensky-network.org/api/states/all"

response = requests.get(states_url, params=zrh_bbox, headers=opensky_auth.headers(), timeout=10)

if response.status_code == 200:
    data = response.json()
    print(f"Total flights found: {len(data['states'])}")
    print("\nSample plane:")
    print(data['states'][0])
else:
    print(f"Error {response.status_code}: {response.text}")

--- FETCHING AUTHENTICATED LIVE AIRSPACE ---
Total flights found: 147

Sample plane:
['39de4e', 'TVF15FY ', 'France', 1789229765, 1789229765, 7.3562, 47.2837, 11590.02, False, 216.98, 283.57, 0, None, 12131.04, '1000', False, 0]


In [6]:
import time
import json
import os
from datetime import datetime

print("\n--- FETCHING HISTORICAL ZURICH ARRIVALS ---")
arrivals_url = "https://opensky-network.org/api/flights/arrival"

days = 1
end_time = int(time.time())
begin_time = end_time - (days * 24 * 60 * 60) 

arrival_params = {
    "airport": "LSZH",
    "begin": begin_time,
    "end": end_time
}

arr_response = requests.get(
    arrivals_url, 
    params=arrival_params, 
    headers=opensky_auth.headers(), 
    timeout=10
)

if arr_response.status_code == 200:
    arrivals_data = arr_response.json()
    print(f"Total arrivals found: {len(arrivals_data)}")
    
    # Save raw JSONs to data/raw/ folder
    if len(arrivals_data) > 0:
        raw_dir = "../data/raw" 
        os.makedirs(raw_dir, exist_ok=True)
        
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filepath = os.path.join(raw_dir, f"arrivals_zrh_{timestamp}.json")
        
        with open(filepath, "w") as f:
            json.dump(arrivals_data, f)
            
        print(f"Saved raw JSON to {filepath}")
else:
    print(f"Error {arr_response.status_code}: {arr_response.text}")


--- FETCHING HISTORICAL ZURICH ARRIVALS ---
Total arrivals found: 138
Saved raw JSON to ../data/raw/arrivals_zrh_20260912_184419.json


In [8]:
## Creating a loop to fetch multiple days
total_days = 30
current_end = int(time.time())

for day in range(total_days):
    # Step backward by exactly 1 day per loop iteration
    begin_time = current_end - (1 * 24 * 60 * 60) 

    arrival_params = {
        "airport": "LSZH",
        "begin": begin_time,
        "end": current_end
    }

    arr_response = requests.get(
        arrivals_url, 
        params=arrival_params, 
        headers=opensky_auth.headers(), 
        timeout=10
    )

    if arr_response.status_code == 200:
        arrivals_data = arr_response.json()
        
        # filename based on the window's end time
        date_label = datetime.fromtimestamp(current_end).strftime('%Y%m%d')
        print(f"[{date_label}] Found {len(arrivals_data)} arrivals.")
        
        if len(arrivals_data) > 0:
            raw_dir = "../data/raw" 
            os.makedirs(raw_dir, exist_ok=True)
            
            filepath = os.path.join(raw_dir, f"arrivals_zrh_{date_label}.json")
            with open(filepath, "w") as f:
                json.dump(arrivals_data, f)
    else:
        print(f"Error {arr_response.status_code}: {arr_response.text}")

    # Shift the window back for the next day
    current_end = begin_time
    
    # Polite delay
    time.sleep(1)

print(f"\n Finished fetching {total_days} days of arrivals data.")


[20260912] Found 135 arrivals.
[20260911] Found 469 arrivals.
[20260910] Found 422 arrivals.
[20260909] Found 430 arrivals.
[20260908] Found 401 arrivals.
[20260907] Found 458 arrivals.
[20260906] Found 420 arrivals.
[20260905] Found 433 arrivals.
[20260904] Found 433 arrivals.
[20260903] Found 435 arrivals.
[20260902] Found 442 arrivals.
[20260901] Found 420 arrivals.
[20260831] Found 425 arrivals.
[20260830] Found 437 arrivals.
[20260829] Found 408 arrivals.
[20260828] Found 399 arrivals.
[20260827] Found 446 arrivals.
[20260826] Found 414 arrivals.
[20260825] Found 401 arrivals.
[20260824] Found 448 arrivals.
[20260823] Found 410 arrivals.
[20260822] Found 421 arrivals.
[20260821] Found 446 arrivals.
[20260820] Found 418 arrivals.
[20260819] Found 419 arrivals.
[20260818] Found 420 arrivals.
[20260817] Found 441 arrivals.
[20260816] Found 410 arrivals.
[20260815] Found 441 arrivals.
[20260814] Found 441 arrivals.

 Finished fetching 30 days of arrivals data.


#### Historical Departures logs

In [ ]:
import time
import json
import os
from datetime import datetime, timedelta, timezone

print("\n--- FETCHING HISTORICAL ZURICH DEPARTURES ---")
departures_url = "https://opensky-network.org/api/flights/departure"

total_days = 30 
current_end = int(time.time())

for day in range(total_days):
    begin_time = current_end - (1 * 24 * 60 * 60) 

    params = {
        "airport": "LSZH",
        "begin": begin_time,
        "end": current_end
    }

    response = requests.get(
        departures_url, 
        params=params, 
        headers=opensky_auth.headers(), 
        timeout=10
    )

    if response.status_code == 200:
        departures_data = response.json()
        date_label = datetime.fromtimestamp(current_end, tz=timezone.utc).strftime('%Y%m%d')
        print(f"[{date_label}] Found {len(departures_data)} departures.")
        
        if len(departures_data) > 0:
            raw_dir = "../data/raw/departures" 
            os.makedirs(raw_dir, exist_ok=True)
            
            filepath = os.path.join(raw_dir, f"departures_zrh_{date_label}.json")
            with open(filepath, "w") as f:
                json.dump(departures_data, f)
    else:
        print(f"Error {response.status_code}: {response.text}")

    current_end = begin_time
    time.sleep(1)

print("\n Finished fetching historical departures.")


--- FETCHING HISTORICAL ZURICH DEPARTURES ---
[20260912] Found 132 departures.
[20260911] Found 459 departures.
[20260910] Found 427 departures.
[20260909] Found 421 departures.
[20260908] Found 401 departures.
[20260907] Found 454 departures.
[20260906] Found 417 departures.
[20260905] Found 421 departures.
[20260904] Found 456 departures.
[20260903] Found 434 departures.
[20260902] Found 429 departures.
[20260901] Found 408 departures.
[20260831] Found 444 departures.
[20260830] Found 436 departures.
[20260829] Found 426 departures.
[20260828] Found 414 departures.
[20260827] Found 422 departures.
[20260826] Found 408 departures.
[20260825] Found 389 departures.
[20260824] Found 454 departures.
[20260823] Found 408 departures.
[20260822] Found 429 departures.
[20260821] Found 438 departures.
[20260820] Found 418 departures.
[20260819] Found 431 departures.
[20260818] Found 412 departures.
[20260817] Found 430 departures.
[20260816] Found 427 departures.
[20260815] Found 426 departur